# Chapitre 13 - Exercices 13.1 à 13.4

* **13.1** : K-means et EM pour un mélange gaussien à covariance $\sigma I$ ; limite $\sigma \to 0$.
* **13.2** : rayon médian du 1-voisinage (éq. (13.7)).
* **13.3** : le théorème de Cover et Hart (éq. (13.5) et (13.12)).
* **13.4** : transformations affines d'une image et vecteurs tangents.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gammaln
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl import ch13_prototypes as cp
set_style()
PURPLE = COLORS[6]

## Exercice 13.1 : K-means et EM pour un mélange à covariance scalaire

**Énoncé (reformulé).** On considère un mélange gaussien dont toutes les matrices de
covariance sont scalaires, $\Sigma_r = \sigma I$ pour $r = 1, \dots, R$, où $\sigma > 0$ est
**fixé** (connu). Discuter en détail l'analogie entre l'algorithme de regroupement K-means
et l'algorithme EM pour ajuster ce mélange. Montrer que lorsque $\sigma \to 0$, les deux
méthodes coïncident.

### Rappels

**Le modèle.** La densité est
$$f(x) = \sum_{r=1}^R \pi_r\, \phi_\sigma(x; \mu_r), \qquad
\phi_\sigma(x; \mu) = (2\pi\sigma)^{-p/2} \exp\Big(-\frac{\|x - \mu\|^2}{2\sigma}\Big),$$
avec des proportions $\pi_r > 0$, $\sum_r \pi_r = 1$. Ici $\sigma$ joue le rôle d'une
**variance** (c'est la notation du livre : $\Sigma_r = \sigma I$).

**K-means.** Minimise $J(C, \mu) = \sum_i \|x_i - \mu_{C(i)}\|^2$ en alternant
affectation au centre le plus proche et recalcul des centres comme moyennes.

**EM** (vu dans le cours 1, section 5.2) : on alterne le calcul des responsabilités
(étape E) et une estimation pondérée des paramètres (étape M) ; chaque itération fait
croître la log-vraisemblance $\ell(\mu, \pi) = \sum_i \log f(x_i)$.

### Les deux étapes d'EM pour ce modèle

**Étape E.** Par la formule de Bayes, la responsabilité de la composante $r$ pour
l'observation $i$ est la probabilité a posteriori que $x_i$ provienne de $r$ :
$$\gamma_{ir} = \frac{\pi_r\, \phi_\sigma(x_i; \mu_r)}{\sum_{s} \pi_s\, \phi_\sigma(x_i; \mu_s)}
= \frac{\pi_r \exp(-d_{ir}/2\sigma)}{\sum_s \pi_s \exp(-d_{is}/2\sigma)}, \qquad d_{ir} = \|x_i - \mu_r\|^2,$$
car le facteur $(2\pi\sigma)^{-p/2}$, commun à toutes les composantes, se simplifie.

**Étape M.** On maximise $\sum_i \sum_r \gamma_{ir}\big[\log \pi_r + \log \phi_\sigma(x_i; \mu_r)\big]$.
En $\mu_r$, seul compte $-\sum_i \gamma_{ir}\|x_i - \mu_r\|^2/(2\sigma)$, fonction concave
dont le gradient $\sum_i \gamma_{ir}(x_i - \mu_r)/\sigma$ s'annule en
$$\mu_r = \frac{\sum_i \gamma_{ir} x_i}{\sum_i \gamma_{ir}} .$$
En $\pi$ (sous la contrainte $\sum_r \pi_r = 1$, par un multiplicateur de Lagrange) :
$\pi_r = \frac1N \sum_i \gamma_{ir}$. Comme $\sigma$ est fixé, il n'est pas ré-estimé.

### L'analogie, étape par étape

| | K-means | EM, $\Sigma_r = \sigma I$ |
|---|---|---|
| affectation | $\gamma_{ir} = \mathbb 1\{r = \arg\min_s d_{is}\}$ (dure) | $\gamma_{ir} \propto \pi_r e^{-d_{ir}/2\sigma}$ (souple) |
| centres | moyenne du groupe | moyenne **pondérée** par les $\gamma_{ir}$ |
| critère | $J$ décroît | $\ell$ croît |

Les deux étapes du K-means sont exactement celles d'EM où l'on aurait remplacé les
responsabilités par des poids 0/1. On peut même relier les critères : si l'on fixe des
affectations dures et $\pi_r = 1/R$, la log-vraisemblance « complète »
$\sum_i \log\big[\pi_{C(i)}\phi_\sigma(x_i; \mu_{C(i)})\big]$ vaut
$-\frac{1}{2\sigma}J(C, \mu) + \text{constante}$ : maximiser l'une revient à minimiser l'autre.

Le paramètre $\sigma$ règle la « dureté » : la responsabilité dépend des **écarts**
de distances au carré, mesurés en unités de $2\sigma$.

### La limite $\sigma \to 0$

Fixons $i$ et supposons que le centre le plus proche de $x_i$, noté $r^*$, soit unique
(c'est le cas sauf pour des points exactement à égale distance de deux centres). En
divisant numérateur et dénominateur par $\pi_r e^{-d_{ir}/2\sigma}$,
$$\gamma_{ir} = \Big(1 + \sum_{s \ne r} \frac{\pi_s}{\pi_r} \exp\Big(-\frac{d_{is} - d_{ir}}{2\sigma}\Big)\Big)^{-1}.$$

* Si $r = r^*$ : pour tout $s \ne r^*$, $d_{is} - d_{ir^*} > 0$, donc
  $\exp(-(d_{is} - d_{ir^*})/2\sigma) \to 0$ quand $\sigma \to 0$ ; ainsi $\gamma_{ir^*} \to 1$.
* Si $r \ne r^*$ : le terme $s = r^*$ vérifie $d_{ir^*} - d_{ir} < 0$, donc
  $\exp(-(d_{ir^*} - d_{ir})/2\sigma) \to +\infty$ ; ainsi $\gamma_{ir} \to 0$.

(Les proportions $\pi_s/\pi_r$ sont des constantes strictement positives : elles ne
changent pas ces limites.) L'étape E devient donc **l'affectation au centre le plus
proche**, et l'étape M devient alors **le calcul des moyennes des groupes** : une
itération d'EM tend vers une itération de Lloyd.

Partant des mêmes centres initiaux, et par continuité (une composition d'un nombre fini
d'applications continues en $\sigma$ hors des cas d'égalité), les $T$ premières
itérations d'EM convergent vers celles du K-means : **les deux méthodes coïncident à la
limite**.

### Vérification numérique

Visualisons d'abord, en dimension 1 avec deux centres $\mu_1 = -1$ et $\mu_2 = 1$, la
responsabilité $\gamma_{\cdot 1}(x)$ pour plusieurs valeurs de $\sigma$ : elle tend vers
l'indicatrice $\mathbb 1\{x < 0\}$ (affectation du K-means).